# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and field/column IDs using Croissant `@id` conventions.

In [ ]:
# List all record sets and their fields/columns with their Croissant @id
print("Available record sets (by @id):")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.id}: {field.name} (dataType: {field.data_type})")

Let's preview a few records from each record set using their Croissant `@id`.

In [ ]:
# Inspect the first few records from each record set (by @id)
for record_set in metadata.record_sets:
    print(f"---\nSample records from RecordSet @id='{record_set.id}':")
    for i, record in enumerate(dataset.records(record_set=record_set.id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. All entities are referenced by their `@id` fields, as shown above.

In [ ]:
# Extract data from each record set (using @id references)
record_set_ids = [r.id for r in metadata.record_sets]
dataframes = {}

# Load DataFrame for each record set
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print columns of the first record set as example
if record_set_ids:
    print(f"Available columns for record set '{record_set_ids[0]}':")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display_df = dataframes[record_set_ids[0]].head()
    display(display_df)

## 4. Exploratory Data Analysis (EDA)
Apply core EDA steps: filtering, normalization, and grouping, referencing all fields by `@id`.

In [ ]:
# Choose one record set with numeric data for analysis. Here, select the first one with a numeric field.
selected_record_set = None
numeric_field_id = None
group_field_id = None

# Look for numeric fields (schema:Float or schema:Integer dataType)
for record_set in metadata.record_sets:
    for field in record_set.fields:
        if field.data_type in ["schema:Float", "schema:Integer", "Float", "Integer"]:
            selected_record_set = record_set
            numeric_field_id = field.id
            break
    if selected_record_set is not None:
        break

# Try to select a potential group field (typically "description" or "gene name", etc.)
if selected_record_set is not None and len(selected_record_set.fields) > 1:
    for field in selected_record_set.fields:
        if field.data_type == "schema:Text" and field.id != numeric_field_id:
            group_field_id = field.id
            break

print(f"Selected record set: {selected_record_set.id if selected_record_set else None}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Group field @id: {group_field_id}")

# Perform filtering, normalization, and grouping
if selected_record_set and numeric_field_id in dataframes[selected_record_set.id].columns:
    df = dataframes[selected_record_set.id]
    # Drop NAs or non-numeric values (in case of string import)
    df_filtered = df.copy()
    df_filtered = df_filtered[pd.to_numeric(df_filtered[numeric_field_id], errors='coerce').notna()]
    df_filtered[numeric_field_id] = pd.to_numeric(df_filtered[numeric_field_id])
    
    threshold = df_filtered[numeric_field_id].mean()
    filtered_df = df_filtered[df_filtered[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
    display(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Optional grouping
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. In this example, we plot a histogram of the selected numeric field and, if applicable, the top groups by mean value.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set and numeric_field_id in dataframes[selected_record_set.id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[selected_record_set.id][numeric_field_id].dropna().astype(float), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in RecordSet '{selected_record_set.id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # If grouping available, show top groups
    if group_field_id and group_field_id in dataframes[selected_record_set.id].columns:
        group_df = dataframes[selected_record_set.id].copy()
        group_df[numeric_field_id] = pd.to_numeric(group_df[numeric_field_id], errors='coerce')
        top_groups = group_df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(10, 6))
        sns.barplot(y=top_groups.index, x=top_groups.values)
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a Croissant-annotated dataset using the `mlcroissant` library. All fields, record sets, and columns were referenced by their unique Croissant `@id` to maintain consistency and reproducibility. The initial overview, data extraction, filtering, normalization, grouping, and basic visualization steps can be extended further for in-depth biological and statistical analysis.